# 03. Evasion Analysis

This notebook demonstrates the **robustness and evasion phase** of the project.

The main question is no longer "what circuit exists?" but:
- what happens to that circuit under conservative, runnable obfuscation?

In mechanistic terms:
- Does obfuscation remove the validated route?
- Or does the model still represent the malicious evidence internally, while changing how it uses that evidence downstream?

In plain language:
- Is the model actually fooled because it stops seeing the danger?
- Or does it still "notice" the danger, but fail to rely on it by the end?


In [ ]:
from pathlib import Path
import math
import textwrap

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(
    context="talk",
    style="whitegrid",
    palette="deep",
    rc={
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.facecolor": "#FCFCFC",
        "figure.facecolor": "white",
        "grid.color": "#D9DDE3",
        "grid.linewidth": 0.8,
        "axes.edgecolor": "#4A5568",
        "axes.labelcolor": "#1A202C",
        "xtick.color": "#2D3748",
        "ytick.color": "#2D3748",
        "axes.titleweight": "semibold",
    },
)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)


def find_project_root() -> Path:
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scaled_validation.py").exists():
            return candidate
        if (candidate / "mech-interp-circuit" / "artifacts").exists():
            return candidate / "mech-interp-circuit"
    raise FileNotFoundError("Could not find mech-interp-circuit project root from the current working directory.")


PROJECT_ROOT = find_project_root()
ARTIFACTS = PROJECT_ROOT / "artifacts" / "foundation_sec"
print("Project root:", PROJECT_ROOT)
print("Artifacts dir:", ARTIFACTS)


def read_csv(name: str) -> pd.DataFrame:
    path = ARTIFACTS / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def clip_text(text: str, *, width: int = 180) -> str:
    text = str(text).strip().replace("\r\n", "\n")
    text = " ".join(text.split())
    if len(text) <= width:
        return text
    return text[: width - 3] + "..."


def show_barh(df, label_col, value_col, *, title, xlabel, color="#2B6CB0", sort=True):
    plot_df = df.copy()
    if sort:
        plot_df = plot_df.sort_values(value_col, ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(plot_df))))
    sns.barplot(data=plot_df, x=value_col, y=label_col, ax=ax, color=color, orient="h")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    sns.despine(ax=ax, left=False, bottom=False)
    plt.tight_layout()
    return fig, ax


## Step 1: Load the evasion benchmark artifacts

We use:
- the expanded technique-level benchmark summary
- a family-level summary of the strict candidate benchmark slices
- a provisional summary that adds the pure `IEX` slice without changing the strict benchmark
- late-carrier patching on the strongest evasion slice
- slice-specific residual and tracing summaries for the final downstream analysis


In [ ]:
benchmark = read_csv("evasion_candidate_benchmark_summary_v3.csv")
family_benchmark = read_csv("evasion_candidate_family_summary_v3.csv")
benchmark_provisional = read_csv("evasion_candidate_benchmark_summary_provisional_v1.csv")
patch_variant = read_csv("evasion_path_patching_late_invoke_webrequest_variant_v2_h100_summary.csv")
trace_variant = read_csv("evasion_trace_resid_pre13_mean_delta_l12_invoke_webrequest_variant_v2_h100_summary.csv")
resid_pre31_variant = read_csv("evasion_resid_pre31_mean_delta_ablate_l12_top4_invoke_webrequest_variant_v2_h100_summary.csv")

benchmark


## Step 2: Which evasion techniques actually work?

A useful evasion benchmark should not treat every string rewrite as equally meaningful.
Here, the important outcome is whether a conservative, syntax-preserving transform actually flips the model from `BLOCK` to `ALLOW`.

In the current strict candidate run:
- `invoke_webrequest_alias` remains the strongest miss pattern
- `downloadstring_psobject_invoke` still produces a narrower miss pattern
- `downloadfile_psobject_invoke` and `split_quoted_encodedcommand_literal` are now benchmarked too, but do **not** produce misses on their current slices
- pure `IEX` variants now appear in a separate provisional tier, so the strict benchmark stays unchanged while the extra slice is still measured


## Step 2a: What the technique names mean

The benchmark technique ids are compact implementation labels. For a new reader, it is much easier to reason about the results if each method is translated into plain language first.

The main methods shown in this notebook are:
- `invoke_webrequest_alias`: replace the full command name `Invoke-WebRequest` with the shorter built-in alias `iwr`
- `downloadstring_psobject_invoke`: hide a direct `.DownloadString(...)` call behind `PSObject.Methods[...]`
- `downloadfile_psobject_invoke`: do the same for `.DownloadFile(...)`
- `split_quoted_encodedcommand_literal`: replace a quoted `-EncodedCommand` string with concatenated pieces such as `"-Encoded" + "Command "`
- `iex_scriptblock_create`: replace direct `iex ...` execution with `&([scriptblock]::Create(...))`

All of these are meant to preserve behavior while reducing an obvious literal surface that the model may rely on.


In [ ]:
technique_defs = pd.DataFrame(
    [
        ("invoke_webrequest_alias", "Use the alias iwr instead of the full Invoke-WebRequest command name.", "strict"),
        ("downloadstring_psobject_invoke", "Rewrite .DownloadString(...) as a PSObject method lookup plus Invoke.", "strict"),
        ("downloadfile_psobject_invoke", "Rewrite .DownloadFile(...) as a PSObject method lookup plus Invoke.", "strict"),
        ("split_quoted_encodedcommand_literal", "Split a quoted -EncodedCommand literal into concatenated string pieces.", "strict"),
        ("iex_scriptblock_create", "Replace direct iex execution with &([scriptblock]::Create(...)).", "provisional"),
    ],
    columns=["technique_id", "plain_language_definition", "benchmark_tier"],
)

technique_defs


## Step 2b: Seed and variant examples

The next table makes the benchmark concrete by showing one original malicious seed snippet and one semantically equivalent evasion variant snippet for each major method.

This matters because the benchmark is not just "random obfuscation." The reader should be able to inspect the before-and-after strings and see exactly what changed.


In [ ]:
seed_manifest = read_csv("evasion_seed_manifest_v2.csv")
variant_strict = read_csv("evasion_variant_manifest_candidate_v3.csv")
variant_provisional = read_csv("evasion_variant_manifest_candidate_provisional_v1.csv")
variant_all = pd.concat([variant_strict, variant_provisional], ignore_index=True).drop_duplicates(subset=["variant_id"])

example_techniques = [
    "invoke_webrequest_alias",
    "downloadstring_psobject_invoke",
    "downloadfile_psobject_invoke",
    "split_quoted_encodedcommand_literal",
    "iex_scriptblock_create",
]

example_rows = []
for technique_id in example_techniques:
    subset = variant_all[variant_all["technique_id"] == technique_id].copy()
    if subset.empty:
        continue
    row = subset.sort_values(["used_char_len", "filename"]).iloc[0]
    seed_row = seed_manifest[seed_manifest["seed_id"] == row["seed_id"]].iloc[0]
    example_rows.append(
        {
            "technique_id": technique_id,
            "tier": row.get("candidate_tier", "strict"),
            "indicator_family": seed_row["primary_indicator"],
            "seed_example": clip_text(seed_row["content"], width=220),
            "variant_example": clip_text(row["content"], width=220),
        }
    )

pd.DataFrame(example_rows)


In [ ]:
plot_df = benchmark.sort_values("evasion_success_rate", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(plot_df["technique_id"], plot_df["evasion_success_rate"], color="#B83280")
ax.set_title("Evasion Success Rate by Technique")
ax.set_xlabel("Evasion success rate")
plt.tight_layout()
plt.show()

benchmark.sort_values("evasion_success_rate", ascending=False)


## Step 2c: Family-level coverage in the current benchmark

The technique summary is useful, but it hides which indicator families are actually represented.
This table makes the current benchmark boundary explicit:
- `DownloadFile` and `-EncodedCommand` now have strict candidate slices with zero misses
- the strongest misses still come from `Invoke-WebRequest` and the narrower `FromBase64String`-linked `DownloadString` route
- pure `IEX` now sits in a separate provisional tier with zero misses on the current slice


In [ ]:
family_benchmark


## Step 2d: Provisional `IEX` coverage

We do not fold these rows into the strict benchmark because they still lack runtime-side parse validation in the current environment.
But we *do* measure them separately so the benchmark can report what happens on the pure `IEX` slice without silently weakening its main admission rule.


In [ ]:
benchmark_provisional[benchmark_provisional["candidate_tier"] == "provisional_iex"]


## Step 3: The first key surprise

On the strongest failure mode, `invoke_webrequest_alias`, the validated late carrier is still **sufficient**.

That means:
- if we patch the old late bundle back in, we can still push the model toward the malicious decision

In plain language:
- the model still knows how to use the old signal if we force it to
- but in the naturally evaded script, it is no longer relying on that signal in the same way


In [ ]:
patch_variant


## Step 4: Trace the late writer family on the evaded variants

The next question is whether the late carrier disappeared.

To test that, we trace which `Layer 12` heads write most strongly into the slice-specific `resid_pre13` malicious-vs-benign direction for the evaded variants.


In [ ]:
top_trace = trace_variant.head(10).copy()
top_trace["head_label"] = top_trace.apply(lambda row: f"L{int(row['layer'])}H{int(row['head'])}", axis=1)

show_barh(
    top_trace,
    "head_label",
    "mean_delta_projection",
    title="Late Writers Into the Variant-Slice resid_pre13 Direction",
    xlabel="Mean delta projection",
    color="#2C7A7B",
    sort=True,
)
plt.show()

top_trace[["head_label", "mean_delta_projection", "positive_delta_frac"]]


### Interpretation

The usual late writer family does **not** disappear.

The familiar heads are still there:
- `L12H15` remains dominant
- `H5`, `H2`, and `H28` remain in the main positive writer set

So the evasion is **not** well described as "the model no longer carries the malicious evidence."


## Step 5: The final downstream probe

The strongest final test asks where the sign split shows up.

Earlier in the late stage, at `resid_pre13`, the late bundle still writes the familiar malicious-evidence direction even on evaded variants.

Later, at `resid_pre31`, we test whether ablating the same late bundle still moves the late residual in the same direction.


In [ ]:
resid_pre31_variant


## Final interpretation

This is the core robustness result:

- the validated late carrier survives the evasion at `resid_pre13`
- but later blocks transform or compensate for that evidence differently
- by `resid_pre31`, the anti-causal split is already visible in the residual stream itself

In plain language:
- the model still contains the danger signal
- but later computation has changed how much the final answer depends on that signal

That is stronger and more precise than saying the circuit was simply "broken."


## Optional: Rerun notes

These commands are GPU-friendly rather than CPU-friendly in practice, but they document the exact downstream probe used in the final writeup.


In [ ]:
# Example only. Uncomment to rerun the final downstream probe.
#
# !python ../scaled_validation.py batch-residual-direction-intervention \
#     --manifest ../artifacts/foundation_sec/evasion_pair_manifest_invoke_webrequest_variant_v2.csv \
#     --basis-path ../artifacts/foundation_sec/evasion_invoke_webrequest_variant_v2_resid_pre31_contrastive_h100_basis.pt \
#     --basis-label mean_delta \
#     --heads 12.15,12.5,12.4,12.28 \
#     --mode ablate \
#     --device cuda \
#     --torch-dtype float16 \
#     --num-pairs 4 \
#     --allow-zero-indicator-malicious \
#     --output-prefix ../artifacts/demo_evasion_probe
